# Titanic: Inspecting and Cleaning Real-World Data

Real datasets are rarely ready for a model. Values can be hidden behind placeholders, numbers can be stored as text, columns can be mostly empty, and some fields only make sense once you know what they mean in the real world.

This notebook walks through a complete **data-cleaning workflow** on the Titanic passenger dataset (1,309 passengers, 14 columns). Every step is explained: *what* we do, *why* we do it, and *how to check* that it worked.

**Workflow**

```text
Inspect → Fix placeholders → Fix dtypes → Measure missingness → Domain checks
        → Drop features → Imputation → Reinspect → (ready for preprocessing)
```

**Contents**
1. Setup and loading the data
2. First inspection
3. Hidden missing values: the `?` placeholder
4. Data type conversion
5. Measuring missingness
6. Domain checks: what does a missing value *mean*?
7. Categorical features: the `boat` column
8. Dropping features (while keeping the original)
9. Imputation: filling the remaining gaps
10. Final check and where cleaning ends

## Column reference

| Column | Meaning |
|---|---|
| `pclass` | Ticket class: 1 = first, 2 = second, 3 = third |
| `survived` | 1 = survived, 0 = did not survive |
| `name` | Passenger name |
| `sex` | Passenger sex |
| `age` | Age in years (fractions for infants) |
| `sibsp` | Number of siblings/spouses aboard |
| `parch` | Number of parents/children aboard |
| `ticket` | Ticket number |
| `fare` | Fare paid |
| `cabin` | Cabin number |
| `embarked` | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |
| `boat` | Lifeboat identifier (if the passenger got into a lifeboat) |
| `body` | Body identification number (if the body was recovered) |
| `home.dest` | Home / destination |

## 1. Setup and loading the data

In [67]:
#import glob

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

The loader below works both on Kaggle (where input files live under `/kaggle/input/`) and locally (where `Titanic.csv` sits next to the notebook).

In [68]:
# kaggle_paths = glob.glob('/kaggle/input/**/[Tt]itanic.csv', recursive=True)
# DATA_PATH = kaggle_paths[0] if kaggle_paths else 'Titanic.csv'

# df = pd.read_csv(DATA_PATH)
# print(f'Loaded {DATA_PATH}: {df.shape[0]} rows, {df.shape[1]} columns')

df = pd.read_csv("Titanic.csv")


## 2. First inspection

**Rule: look at the data before changing it.** Two tools do two different jobs:

- `df.head()` shows the **values** in the first rows, so you can spot odd entries.
- `df.info()` shows the **structure**: column names, non-null counts, and dtypes.

Neither is enough on its own. The interesting part is comparing them.

In [69]:
df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29,0,0,24160,211.3375,B5,S,2,?,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.55,C22 C26,S,11,?,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2,1,2,113781,151.55,C22 C26,S,?,?,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30,1,2,113781,151.55,C22 C26,S,?,135,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25,1,2,113781,151.55,C22 C26,S,?,?,"Montreal, PQ / Chesterville, ON"


In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   pclass     1309 non-null   int64
 1   survived   1309 non-null   int64
 2   name       1309 non-null   str  
 3   sex        1309 non-null   str  
 4   age        1309 non-null   str  
 5   sibsp      1309 non-null   int64
 6   parch      1309 non-null   int64
 7   ticket     1309 non-null   str  
 8   fare       1309 non-null   str  
 9   cabin      1309 non-null   str  
 10  embarked   1309 non-null   str  
 11  boat       1309 non-null   str  
 12  body       1309 non-null   str  
 13  home.dest  1309 non-null   str  
dtypes: int64(4), str(10)
memory usage: 143.3 KB


**What to notice**

- `info()` reports **1309 non-null** values for *every* column, which suggests no missing data at all.
- But `head()` already shows a `?` in the `body` column, and `age`, `fare`, and `body` are stored as `object` (text) even though they should be numbers.

That mismatch is our first clue: missing values are hiding behind a placeholder.

## 3. Hidden missing values: the `?` placeholder

In this file, missing entries were written as `?` instead of being left empty. To pandas, `?` is just a one-character string, so the cell counts as **non-null**.

- A **missing value** is the real observation that is absent (e.g. an age that was never recorded).
- A **placeholder** is the symbol written in its place (`?`).

**Assumption:** in this dataset `?` always means "missing". In another dataset `?` could be a real value, so always confirm what a placeholder means before replacing it.

We replace every `?` with `np.nan`, pandas' standard missing-value marker, and inspect again.

In [71]:
print('Cells containing "?" before:', (df == '?').sum().sum())

df = df.replace('?', np.nan)

print('Cells containing "?" after: ', (df == '?').sum().sum())

Cells containing "?" before: 3855
Cells containing "?" after:  0


In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   pclass     1309 non-null   int64
 1   survived   1309 non-null   int64
 2   name       1309 non-null   str  
 3   sex        1309 non-null   str  
 4   age        1046 non-null   str  
 5   sibsp      1309 non-null   int64
 6   parch      1309 non-null   int64
 7   ticket     1309 non-null   str  
 8   fare       1308 non-null   str  
 9   cabin      295 non-null    str  
 10  embarked   1307 non-null   str  
 11  boat       486 non-null    str  
 12  body       121 non-null    str  
 13  home.dest  745 non-null    str  
dtypes: int64(4), str(10)
memory usage: 143.3 KB


The non-null counts now tell the truth. For example, `age` drops from 1309 to **1046** non-null values. The dtypes of `age`, `fare`, and `body` are still `object`, because they were read as text. We fix that next.

## 4. Data type conversion

A number stored as text cannot be used as a number: you cannot average it or compare it numerically. A clean numeric feature needs **both** numeric values **and** a numeric dtype.

`pd.to_numeric(..., errors='coerce')` converts a column to numbers:

- values that parse as numbers are converted;
- anything that cannot be parsed becomes `NaN` instead of raising an error.

Coercion does not *fix* bad values; it *exposes* them as missing, so they can be handled later.

**Assumption:** the columns we convert are genuinely numeric quantities. Columns such as `ticket` contain digits but are labels, so they are not converted.

In [73]:
numeric_columns = ['age', 'sibsp', 'parch', 'fare', 'survived', 'pclass', 'body']

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   str    
 3   sex        1309 non-null   str    
 4   age        1046 non-null   float64
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   ticket     1309 non-null   str    
 8   fare       1308 non-null   float64
 9   cabin      295 non-null    str    
 10  embarked   1307 non-null   str    
 11  boat       486 non-null    str    
 12  body       121 non-null    float64
 13  home.dest  745 non-null    str    
dtypes: float64(3), int64(4), str(7)
memory usage: 143.3 KB


`age`, `fare`, and `body` are now `float64`. The non-null counts did not drop, so no unexpected text was hiding in these columns: every non-missing value converted cleanly.

> Integer-like columns that contain `NaN` (such as `body`) become `float64`, because a standard integer column cannot hold `NaN`. That is expected.

## 5. Measuring missingness

Now that every missing value is a real `NaN`, we can measure how much is missing per column.

$$
\text{Missingness \%} = \frac{\text{Number of missing values}}{\text{Total number of records}} \times 100
$$

A percentage puts every column on the same 0–100 scale, so columns (and datasets of different sizes) can be compared fairly.

In [74]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percentage = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_percentage': missing_percentage,
})
missing_summary

,missing_count,missing_percentage
body,1188,90.76
cabin,1014,77.46
boat,823,62.87
home.dest,564,43.09
age,263,20.09
embarked,2,0.15
fare,1,0.08
sibsp,0,0.00
name,0,0.00
survived,0,0.00


**Reading the table**

| Column | Missing | Interpretation (investigated below) |
|---|---|---|
| `body` | ~91% | Only recovered victims have a body number |
| `cabin` | ~77% | Most passengers have no recorded cabin |
| `boat` | ~63% | Only people who got into a lifeboat have one |
| `home.dest` | ~43% | Large gaps in home/destination |
| `age` | ~20% | Moderate gaps in an important feature |
| `embarked` | 2 rows | Nearly complete |
| `fare` | 1 row | Nearly complete |

A percentage is **evidence, not a decision**. A column that is 91% missing is not automatically useless; we first need to ask *why* it is missing.

> `missing_summary` is a diagnostic table. Changing it does not change `df`.

## 6. Domain checks: what does a missing value *mean*?

Some columns record real-world events, so their missingness follows real-world rules (**domain constraints**):

- `boat` is filled only if the passenger got into a lifeboat, which strongly suggests survival.
- `body` is filled only if the passenger died *and* the body was recovered.

Let's test those rules against `survived`.

In [75]:
has_boat = df['boat'].notna()

survival_rate_with_boat = df.loc[has_boat, 'survived'].mean() * 100
survival_rate_without_boat = df.loc[~has_boat, 'survived'].mean() * 100

print(f'Survival rate of passengers WITH a boat:    {survival_rate_with_boat:.2f}%')
print(f'Survival rate of passengers WITHOUT a boat: {survival_rate_without_boat:.2f}%')

Survival rate of passengers WITH a boat:    98.15%
Survival rate of passengers WITHOUT a boat: 2.79%


Be careful with the direction of these percentages:

- **98.15% of boat holders survived.** This is *P(survived | boat)*.
- It does **not** mean 98% of survivors had a boat. That is *P(boat | survived)*, a different question, computed below.

In [76]:
survivors = df[df['survived'] == 1]
share_of_survivors_with_boat = survivors['boat'].notna().mean() * 100

print(f'Survivors: {len(survivors)}')
print(f'Survivors with a boat value: {survivors["boat"].notna().sum()} '
      f'({share_of_survivors_with_boat:.1f}%)')

Survivors: 500
Survivors with a boat value: 477 (95.4%)


About 95% of survivors have a boat value. The remaining survivors are a small gap worth investigating, not ordinary random absence.

Next, the **contradiction check**: a passenger who did *not* survive should not have a lifeboat. Rows that break this rule are candidates for data errors.

In [77]:
died_with_boat = df[has_boat & (df['survived'] == 0)]

print(f'Passengers with a boat value who did not survive: {len(died_with_boat)}')
died_with_boat[['name', 'pclass', 'survived', 'boat', 'body']]

Passengers with a boat value who did not survive: 9


,name,pclass,survived,boat,body
19,"Beattie, Mr. Thomson",1,0,A,NaN
166,"Hoyt, Mr. William Fisher",1,0,14,NaN
544,"Renouf, Mr. Peter Henry",2,0,12,NaN
655,"Backstrom, Mr. Karl Alfred",3,0,D,NaN
853,"Harmer, Mr. Abraham (David Lishin)",3,0,B,NaN
921,"Keefe, Mr. Arthur",3,0,A,NaN
968,"Lindell, Mr. Edvard Bengtsson",3,0,A,NaN
969,"Lindell, Mrs. Edvard Bengtsson (Elin Gerda Per...",3,0,A,NaN
1299,"Yasbeck, Mr. Antoni",3,0,C,NaN


These rows contradict the domain rule. They might be data errors, or genuine cases (for example, someone who reached a lifeboat but died later). Either way, they are flagged rather than silently trusted.

Now the `body` column:

In [78]:
has_body = df['body'].notna()
survival_rate_with_body = df.loc[has_body, 'survived'].mean() * 100

print(f'Passengers with a body number: {has_body.sum()}')
print(f'Survival rate of passengers with a body number: {survival_rate_with_body:.2f}%')

Passengers with a body number: 121
Survival rate of passengers with a body number: 0.00%


**No passenger with a body number survived**, exactly as the domain rule predicts. So `body` being ~91% missing is not a data-collection failure: survivors *should* have no body number.

This is why we never fill `body` with a mode or mean; that would invent body numbers for survivors.

**Takeaway:** missingness is information about how the data was produced. Ask what a missing value *means* before deciding how to fix it.

## 7. Categorical features: the `boat` column

`boat` looks partly numeric, so let's inspect its distinct values.

In [79]:
df['boat'].unique()

<StringArray>
[      '2',      '11',       nan,       '3',      '10',       'D',       '4',
       '9',       '6',       'B',       '8',       'A',       '5',       '7',
       'C',      '14',     '5 9',      '13',       '1',      '15',     '5 7',
    '8 10',      '12',      '16', '13 15 B',     'C D',   '15 16',   '13 15']
Length: 28, dtype: str

The values mix numbers (`'2'`, `'11'`), letters (`'D'`, `'A'`), and combinations (`'5 9'`, `'13 15 B'`). These are **labels** naming lifeboats, not quantities: boat 13 is not "more" than boat 4. Converting them to numbers would create a meaningless order.

**Idea:** replace the messy labels with a simple binary feature, `boat_assigned` (1 = has a boat value, 0 = no boat value). Before keeping a new feature, check whether it adds new information:

In [80]:
boat_assigned = df['boat'].notna().astype(int)
pd.crosstab(boat_assigned, df['survived'],
            rownames=['boat_assigned'], colnames=['survived'])

survived,0,1
boat_assigned,,
0,800,23
1,9,477


`boat_assigned` almost perfectly matches `survived`. Two problems follow:

1. **Redundancy:** it repeats information we already have.
2. **Target leakage:** `survived` is what we want to predict. A lifeboat assignment is only known *after* the event, so a model using it would look excellent in testing but be useless for real prediction.

So neither `boat` nor `boat_assigned` is kept as a model feature. A simpler representation is only useful if it adds information that isn't already elsewhere.

## 8. Dropping features (while keeping the original)

| Column | Decision | Reason |
|---|---|---|
| `body` | drop | ~91% missing; only describes recovered victims (outcome information) |
| `cabin` | drop | ~77% missing; too incomplete to clean cheaply |
| `boat` | drop | Labels, not quantities; as a feature it leaks the target |
| `home.dest` | drop | ~43% missing; judged not relevant to survival |
| `age` | **keep** | ~20% missing and plausibly related to survival, so worth repairing |

Dropping is a decision backed by evidence (missingness + meaning + usefulness), never an automatic rule.

We drop the columns into a **new working copy** `df_clean`. The original `df` stays intact, because later checks (like the `body` check above) may need columns that were dropped.

In [81]:
columns_to_drop = ['body', 'cabin', 'boat', 'home.dest']

df_clean = df.drop(columns=columns_to_drop)

print('Original columns:', df.shape[1])
print('Working copy columns:', df_clean.shape[1])
df_clean.isna().sum()

Original columns: 14
Working copy columns: 10


pclass        0
survived      0
name          0
sex           0
age         263
sibsp         0
parch         0
ticket        0
fare          1
embarked      2
dtype: int64

In [82]:
# Reinspect after dropping — confirm dtypes and remaining gaps
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   survived  1309 non-null   int64  
 2   name      1309 non-null   str    
 3   sex       1309 non-null   str    
 4   age       1046 non-null   float64
 5   sibsp     1309 non-null   int64  
 6   parch     1309 non-null   int64  
 7   ticket    1309 non-null   str    
 8   fare      1308 non-null   float64
 9   embarked  1307 non-null   str    
dtypes: float64(2), int64(4), str(4)
memory usage: 102.4 KB


Three columns still have gaps: `age` (263), `embarked` (2), and `fare` (1).

## 9. Imputation: filling the remaining gaps

**Imputation** means replacing missing values using a stated rule. The common choices:

- **Mean:** the average, $\text{Mean} = \frac{1}{n}\sum_{i=1}^{n} x_i$, where $x_i$ are the $n$ observed values. Sensitive to extreme values.
- **Median:** the middle value of the sorted data. Robust to extreme values.
- **Mode:** the most frequent value. The only option for categories.

The fill statistic must match the feature's **type**: a category needs the mode, since you cannot average port names.

> `.mode()` returns a Series (there can be ties), so `.iloc[0]` takes the first mode as a single value.

### 9.1 `age`, filled with its mode

In [83]:
age_mode = df_clean['age'].mode().iloc[0]
print('Most frequent age:', age_mode)

df_clean['age'] = df_clean['age'].fillna(age_mode)

Most frequent age: 24.0


> **Trade-off:** filling ~20% of ages with a single value creates an artificial spike at that age in the distribution. The median is a common alternative for numeric features; the mode is used here as the chosen rule.

### 9.2 `embarked`, filled with its own mode

`embarked` is categorical, so the mode is the natural fill. It must be the mode **of `embarked` itself**: filling it with another column's statistic (for example the age mode, 24.0) would inject a number into a column of port letters.

In [84]:
embarked_mode = df_clean['embarked'].mode().iloc[0]
print('Most frequent port:', embarked_mode)

df_clean['embarked'] = df_clean['embarked'].fillna(embarked_mode)
print('Distinct ports after filling:', df_clean['embarked'].unique())

Most frequent port: S
Distinct ports after filling: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str


Only valid port codes remain.

### 9.3 `fare`, filled with its own mode

Only one value is missing, so the choice barely affects the column. As with `embarked`, the fill comes from `fare` itself.

In [85]:
fare_mode = df_clean['fare'].mode().iloc[0]
print('Most frequent fare:', fare_mode)

df_clean['fare'] = df_clean['fare'].fillna(fare_mode)

Most frequent fare: 8.05


## 10. Final check and where cleaning ends

Always **reinspect** after cleaning: confirm the dtypes are right and no missing values remain.

In [86]:
print('Missing values per column:')
print(df_clean.isna().sum())
print()
df_clean.info()

Missing values per column:
pclass      0
survived    0
name        0
sex         0
age         0
sibsp       0
parch       0
ticket      0
fare        0
embarked    0
dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   survived  1309 non-null   int64  
 2   name      1309 non-null   str    
 3   sex       1309 non-null   str    
 4   age       1309 non-null   float64
 5   sibsp     1309 non-null   int64  
 6   parch     1309 non-null   int64  
 7   ticket    1309 non-null   str    
 8   fare      1309 non-null   float64
 9   embarked  1309 non-null   str    
dtypes: float64(2), int64(4), str(4)
memory usage: 102.4 KB


In [87]:
df_clean.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,embarked
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,S
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,S
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,S
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,S
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,S


The working copy now has 10 columns, correct dtypes, and **no missing values**.

### Cleaning vs. preprocessing

Cleaning fixed **quality** problems: placeholders, wrong dtypes, missingness, uninformative or leaking features, and gaps. The data is still not *model-ready*. **Preprocessing** comes next and changes the data's **form**, for example turning `sex` and `embarked` into numbers (encoding) and putting numeric features on similar scales (scaling).

### Summary

| Step | What we did | Why |
|---|---|---|
| Inspect | `head()` + `info()` | `info()` alone hid the `?` placeholders |
| Placeholders | `?` → `NaN` | Make missingness visible to pandas |
| Dtypes | `pd.to_numeric(errors='coerce')` | Numbers stored as text can't be used as numbers |
| Missingness | Count + % per column | Evidence for decisions |
| Domain checks | `boat`/`body` vs `survived` | Missingness can carry meaning; found 9 contradictory rows |
| Categorical | `boat` labels → `boat_assigned` tested | Redundant with the target, so not kept |
| Drop | `body`, `cabin`, `boat`, `home.dest` | Evidence + meaning, on a working copy |
| Impute | `age`, `embarked`, `fare` with their own modes | Fill rule must match the feature |
| Reinspect | No missing values left | Confirm the cleaning worked |

A model cannot compensate for misunderstood or inconsistent input data, which is why cleaning is such a large part of real machine-learning work.